In [1]:
!pip install ultralytics -q
!pip install gdown==4.6.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s eta 0:00:0000:01


In [2]:
# https://drive.google.com/file/d/1sqLnUpQi5eUO9-f65IAkOZQlSh-_ZLKz/view?usp=sharing
!gdown 1sqLnUpQi5eUO9-f65IAkOZQlSh-_ZLKz

Downloading...
From: https://drive.google.com/uc?id=1sqLnUpQi5eUO9-f65IAkOZQlSh-_ZLKz
To: /kaggle/working/best.pt
100%|██████████████████████████████████████| 22.5M/22.5M [00:00<00:00, 77.9MB/s]


In [3]:
!git clone https://github.com/nwojke/deep_sort.git

Cloning into 'deep_sort'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 167 (delta 4), reused 3 (delta 3), pack-reused 148 (from 2)
Receiving objects: 100% (167/167), 86.11 KiB | 6.62 MiB/s, done.
Resolving deltas: 100% (85/85), done.


In [4]:
#https://drive.google.com/drive/folders/18fKzfqnqhqW3s9zwsCbnVJ5XF2JFeqMp
!gdown --no-check-certificate --folder https://drive.google.com/drive/folders/18fKzfqnqhqW3s9zwsCbnVJ5XF2JFeqMp

Retrieving folder list
Retrieving folder 1VVqtL0klSUvLnmBKS89il1EKC3IxUBVK detections
Retrieving folder 1qNWOpUtKG8GqEiL-LbBdXyvifUtcbOvc MOT16_POI_test
Processing file 1aEzvFHPK-N6hqLXMqhh3i9JJzn7WFUA3 MOT16-01.npy
Processing file 1h_ktJDBIEXaSBAA-RxKNYnL9e4fp2HPd MOT16-03.npy
Processing file 1ilOElwfYZLwQKH57HoYdXfuYhpPibfqF MOT16-06.npy
Processing file 1TajzH3GbumKmtYvKBvOtGERFGD0tStwG MOT16-07.npy
Processing file 1WB9Mi4RLVPHV4_20sVq7FdoeG5JYQ_J1 MOT16-08.npy
Processing file 1mksH9GWNT7zmcuq6rlRev8pevZz8Rfsm MOT16-12.npy
Processing file 1FVVhn_IpxQ_jkYhc0CUQHSQMm1SMTEBj MOT16-14.npy
Retrieving folder 1DcOcApOkxP3NdeIUXxVF1KNex6T6YDq3 MOT16_POI_train
Processing file 1Va__9NWU2ZCmaxIq4oIabi05NYWEOk1K MOT16-02.npy
Processing file 1EH7orgDPp7kqRY5OA0hEctcEtQnYq0Ea MOT16-04.npy
Processing file 1RCfHJx5ZoUecapbZCsgp0tCEiItvLsd8 MOT16-05.npy
Processing file 1VLOvn-mbpY0Q1rsMONQZhaEQIGEmyLQL MOT16-09.npy
Processing file 1SbMhOgYPvZ84xE8lRtXc7CLXJF86lwf4 MOT16-10.npy
Processing file 1a4w-Ho

In [17]:
import os
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [18]:
from ultralytics import YOLO

class Detector:
    def __init__(self, model_path):
        self.model = YOLO(model_path)

    def detect(self, src_img):
        results = self.model(src_img)[0]

        bboxes = results.boxes.xywh.cpu().numpy()
        bboxes[:, :2] = bboxes[:, :2] - (bboxes[:, 2:] / 2)

        scores = results.boxes.conf.cpu().numpy()
        class_ids = results.boxes.cls.cpu().numpy()

        return bboxes, scores, class_ids

In [58]:
from deep_sort.deep_sort import nn_matching
from deep_sort.deep_sort.detection import Detection
from deep_sort.deep_sort.tracker import Tracker
from deep_sort.tools import generate_detections as gdet

class deepSORT:
    def __init__(
        self,
        model_path='resources/networks/mars-small128.pb',
        max_cosine_distance=0.7,
        nn_budget=None,
        classes=['objects']
    ):

        self.encoder = gdet.create_box_encoder(model_path, batch_size=1)

        self.metric = nn_matching.NearestNeighborDistanceMetric(
            'cosine',
            max_cosine_distance,
            nn_budget
        )

        self.tracker = Tracker(self.metric)

        key_list = []
        val_list = []

        for ID, class_name in enumerate(classes):
            key_list.append(ID)
            val_list.append(class_name)

        self.key_list = key_list
        self.val_list = val_list
        
    def tracking(self, origin_frame, bboxes, scores):
    
        features = self.encoder(origin_frame, bboxes)
    
        detections = [ Detection(bbox, score, feature) for bbox, score, feature in zip(bboxes, scores, features) ]
    
        self.tracker.predict()
        self.tracker.update(detections)
    
        tracked_bboxes = []
    
        for track in self.tracker.tracks:
            if not track.is_confirmed() or track.time_since_update > 5:
                continue
    
            bbox = track.to_tlbr()
            tracking_id = track.track_id
    
            tracked_bboxes.append(
                bbox.tolist() + [tracking_id]
            )
    
        tracked_bboxes = np.array(tracked_bboxes)
    
        return tracked_bboxes

In [60]:
def draw_detection(img, bboxes, scores, ids, mask_alpha=0.3):
    height, width = img.shape[:2]
    np.random.seed(0)
    rng = np.random.default_rng(3)
    colors = rng.uniform(0, 255, size=(100, 3))
    
    mask_img = img.copy()
    det_img = img.copy()
    
    size = min([height, width]) * 0.0006
    text_thickness = int(min([height, width]) * 0.001)

    # Draw bounding boxes and labels of detections
    for bbox, score, id_ in zip(bboxes, scores, ids):
        color = colors[id_ % 100]
    
        x1, y1, x2, y2 = bbox.astype(int)
    
        # Draw rectangle
        cv2.rectangle(det_img, (x1, y1), (x2, y2), color, 2)
    
        # Draw fill rectangle in mask image
        cv2.rectangle(mask_img, (x1, y1), (x2, y2), color, -1)
    
        caption = f'ID {id_} {int(score*100)}%'
    
        (tw, th), _ = cv2.getTextSize(
            text=caption,
            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
            fontScale=size,
            thickness=text_thickness
        )
    
        th = int(th * 1.2)
    
        cv2.rectangle(det_img, (x1, y1), (x1 + tw, y1 - th), color, -1)
        cv2.rectangle(mask_img, (x1, y1), (x1 + tw, y1 - th), color, -1)
    
        cv2.putText(
            det_img,
            caption,
            (x1, y1),
            cv2.FONT_HERSHEY_SIMPLEX,
            size,
            (255, 255, 255),
            text_thickness,
            cv2.LINE_AA
        )
    
        cv2.putText(
            mask_img,
            caption,
            (x1, y1),
            cv2.FONT_HERSHEY_SIMPLEX,
            size,
            (255, 255, 255),
            text_thickness,
            cv2.LINE_AA
        )
    
    return cv2.addWeighted(mask_img, mask_alpha, det_img, 1 - mask_alpha, 0)

In [61]:
def video_tracking(video_path,detector,tracker,is_save_result=False,save_dir='tracking_results'):
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    if is_save_result:
        os.makedirs(save_dir, exist_ok=True)
    
        # Get the video properties
        fps = int(cap.get(cv2.CAP_PROP_FPS))
    
        # Define the codec and create the video writer
        fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    
        save_result_name = 'output_video2.avi'
        save_result_path = os.path.join(save_dir, save_result_name)
    
        out = cv2.VideoWriter(save_result_path, fourcc, fps, (width, height))

    all_tracking_results = []
    tracked_ids = np.array([], dtype=np.int32)
    
    while True:
        ret, frame = cap.read()
    
        if not ret:
            break
    
        detector_results = detector.detect(frame)
        bboxes, scores, class_ids = detector_results
    
        tracker_pred = tracker.tracking(
            origin_frame=frame,
            bboxes=bboxes,
            scores=scores
        )
    
    
        if tracker_pred.size > 0:
            bboxes = tracker_pred[:, :4]
            tracking_ids = tracker_pred[:, 4].astype(int)
            
            conf_scores = scores[:len(bboxes)]
        
            # Get new tracking IDs
            new_ids = np.setdiff1d(tracking_ids, tracked_ids)
        
            # Store new tracking IDs
            tracked_ids = np.concatenate((tracked_ids, new_ids))
        
            result_img = draw_detection(
                img=frame,
                bboxes=bboxes,
                scores=conf_scores,
                ids=tracking_ids
            )
        else:
            result_img = frame

        all_tracking_results.append(tracker_pred)
    
        if is_save_result == 1:
            out.write(result_img)
        
        # Break the loop if 'q' is pressed
        if cv2.waitKey(25) & 0xFF == ord('q'):
            break
    
    # Release video capture
    cap.release()
    if is_save_result:
        out.release()
    
    cv2.destroyAllWindows()
    
    return all_tracking_results

In [62]:
yolo_model_path = "/kaggle/working/best.pt"
detector = Detector(yolo_model_path)
tracker = deepSORT()

I0000 00:00:1773640294.794598      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1773640294.796095      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [55]:
#https://drive.google.com/file/d/1z1KyE2gXSEQrUHmgX5Jjha8ONC_rVG3R/view?usp=sharing
#https://drive.google.com/file/d/1M--7OwuYB6OEoP3M9dKvtlHGaVJqraAZ/view?usp=sharing
!gdown 1M--7OwuYB6OEoP3M9dKvtlHGaVJqraAZ

Downloading...
From: https://drive.google.com/uc?id=1M--7OwuYB6OEoP3M9dKvtlHGaVJqraAZ
To: /kaggle/working/pedestrian_s2.mp4
100%|██████████████████████████████████████| 11.8M/11.8M [00:00<00:00, 95.6MB/s]


In [63]:
video_path = '/kaggle/working/pedestrian_s2.mp4'
all_tracking_results = video_tracking(
    video_path,
    detector,
    tracker,
    is_save_result=True
)


0: 384x640 8 objectss, 11.4ms
Speed: 2.3ms preprocess, 11.4ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 2.3ms preprocess, 10.7ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 3.0ms preprocess, 10.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 2.3ms preprocess, 10.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 2.1ms preprocess, 10.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 2.1ms preprocess, 10.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 3.1ms preprocess, 10.7ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 objectss, 10.7ms
Speed: 2.1ms preprocess, 10.7ms inference, 1.3ms postprocess per image at